In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry

l = 2 #m halbe Länge
b = 1 #m halbe Breite

radius = 0.025 # Radius der Kreise für Auflager
 
shape = MoveTo(0,0).Rectangle(l,b).Face()

shape.edges.name="free"

#circle = Circle((0,0),0.10).Face()
#circle.edges.name="circles"

auflager_liste = [(radius,radius),(l-radius,b-radius),(radius,b-radius),(l-radius,radius)]

# auflager_liste = [(-a+radius,-b+radius),(-a/2,b/2),(-a/2,-b/2),(a/2,-b/2)]
#auflager_liste = [(-0.3,-0.1),(-0.1,0)]

rect = shape 
for mittelpunkt in auflager_liste:
    circle = Circle(mittelpunkt,radius).Face()
    circle.edges.name = "dirichlet"
    rect = rect - circle

geo = OCCGeometry(rect,dim=2)

#Draw(rect)
mesh = Mesh(geo.GenerateMesh(maxh=l/20))

mesh.Curve(3)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [2]:
E  = 70e9      #Glas ~ 70 GPa Elastizitätsmodul
nu = 0.22
t  = 0.003     #Dicke [m]
rho = 2500   #kg/m^3 Dichte einer Aluminiumplatte
g = 9.81

q = rho * t * g     #Eigengewicht der Platte - rechte Seite der PDE

Db = E*t**3/(12*(1-nu**2))

def D(A):
    return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

def Dinv(A):
    return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))



In [3]:
order = 3

V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
Q = H1(mesh, order=order, dirichlet="dirichlet")
X = V * Q

(sigma,w),(tau,v)= X.TnT()

n = specialcf.normal(2)

def tang(u):
    return u - (u*n)*n

a = BilinearForm(X,symmetric=True)
a += InnerProduct(Dinv(sigma),tau) * dx
a += div(sigma)*Grad(v) * dx
a += div(tau) * Grad(w) * dx
a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
a.Assemble()

L = LinearForm(X)
L += q * v *dx
L.Assemble()

gf_solution = GridFunction(X)
gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

gf_sigma, gf_w = gf_solution.components
#Draw(gf_sigma, mesh,name="sigma")
Draw(gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])




WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

BaseWebGuiScene

In [18]:
import numpy as np
np.set_printoptions(threshold=np.inf)

step = 40

# 1. Gitterpunkte erzeugen (ersetze x_min, x_max, y_min, y_max durch deine Werte)
x_points = np.linspace(0,l, step)
y_points = np.linspace(0,b, step)
X, Y = np.meshgrid(x_points, y_points)

# 2. Ergebnisse berechnen (passe die Auswertung an deine gf_w-Struktur an)
result_matrix = np.zeros((step, step),dtype=object)
for i in range(step):
    for j in range(step):
        # Beispiel: gf_w(X[i, j], Y[i, j]) – ggf. anpassen!
        result_matrix[i, j] = (float(X[i,j]), float(Y[i,j]), gf_w(X[i, j], Y[i, j]))

# Jetzt enthält result_matrix die gewünschten Werte

print(result_matrix)


# Speichern als CSV-Datei
with open("ergebnisse.txt", "w") as f:
    for i in range(step):
        for j in range(step):
            x, y, w = result_matrix[i, j]
            f.write(f"{x:.6f}\t{y:.6f}\t{w:.6e}\n")




[[(0.0, 0.0, -3.3194945791217933e-10)
  (0.05128205128205128, 0.0, -3.141477665565867e-05)
  (0.10256410256410256, 0.0, -0.0006657842781606307)
  (0.15384615384615385, 0.0, -0.0016768597013426217)
  (0.20512820512820512, 0.0, -0.00291134204654947)
  (0.2564102564102564, 0.0, -0.0042989402172778106)
  (0.3076923076923077, 0.0, -0.0057875018369071786)
  (0.358974358974359, 0.0, -0.00733380496601766)
  (0.41025641025641024, 0.0, -0.008899942193801596)
  (0.4615384615384615, 0.0, -0.010452382306150265)
  (0.5128205128205128, 0.0, -0.011960629028103414)
  (0.5641025641025641, 0.0, -0.013397388831731118)
  (0.6153846153846154, 0.0, -0.014737845987516414)
  (0.6666666666666666, 0.0, -0.015960113658284602)
  (0.717948717948718, 0.0, -0.017044691709778938)
  (0.7692307692307692, 0.0, -0.017975018825839583)
  (0.8205128205128205, 0.0, -0.018736964740090047)
  (0.8717948717948718, 0.0, -0.019319412219171182)
  (0.923076923076923, 0.0, -0.019713743543160137)
  (0.9743589743589743, 0.0, -0.01991441